# NOISE-01 · CP-PLL 分源相噪、抖动与 TI EVM 对照

同一计量口径下计算七源输出单边 PSD，区分连续随机相噪与离散纯 PM。TI SNAU210B Fig. 5-1 的标记另做固定留出验证。教学参数与实测读数分别保留。

Python 3.12，NumPy 2.3.5 和 Matplotlib 3.10.6。初次运行需下载固定版本公开数据和脚本。

In [1]:
from pathlib import Path
from urllib.request import urlopen
import csv
import json
import math
import subprocess
import sys
import tempfile

revision = 'aedf3102f710af80702e6aa8eada9fad3bf0e88a'
root = f'https://raw.githubusercontent.com/thecloud723/analog-ic-learning/{revision}/pll/labs/NOISE-01'
workspace = Path(tempfile.mkdtemp(prefix='noise01-', dir=Path.cwd()))
lab = workspace / 'source'
lab.mkdir()
for name in ['noise01.py', 'check_noise01.py', 'verify_noise01_fft.py',
             'noise01-settings.json', 'compare_evm.py', 'ti-evm-markers.json']:
    with urlopen(f'{root}/{name}', timeout=45) as response:
        (lab / name).write_bytes(response.read())
cfg = lab / 'noise01-settings.json'
out = workspace / 'results'
print('Source commit:', revision)

Source commit: aedf3102f710af80702e6aa8eada9fad3bf0e88a


## 默认预算、时域重建和实测留出

单边相位 PSD 积分直接得到相位方差，线性 SSB 相噪积分需乘 2。合计含纯 PM 时，对独立或正交项在方差域相加。

In [2]:
def execute(script, *args):
    return subprocess.run([sys.executable, lab / script, *args], check=True,
                          capture_output=True, text=True)
execute('check_noise01.py')
execute('noise01.py', cfg, out)
execute('verify_noise01_fft.py', cfg, out / 'fft-check.json')
execute('compare_evm.py', lab / 'ti-evm-markers.json', out / 'evm.json')
summary = json.loads((out / 'summary.json').read_text())
evm = json.loads((out / 'evm.json').read_text())
print('Carrier: %.3f GHz' % (summary['carrier_hz'] / 1e9))
print('12 kHz–20 MHz random RMS: %.6f ps' % (summary['random_rms_time_s'] * 1e12))
print('Separate narrowband PM RMS: %.6f ps' % (summary['pm_spur_rms_time_s'] * 1e12))
print('TI fixed holdout RMS residual: %.6f dB' % evm['validation']['rms_residual_db'])

Carrier: 5.000 GHz
12 kHz–20 MHz random RMS: 0.755425 ps
Separate narrowband PM RMS: 0.014235 ps
TI fixed holdout RMS residual: 3.536118 dB


## 查看模型、传递和实测残差

上图的七源参数是教学假设。下图为 TI 公开仪器标记及经验插值，留出点不参与训练。频偏在对数轴上显示，积分仍对 Hz 进行。

In [3]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
table = np.genfromtxt(out / 'spectrum.csv', delimiter=',', names=True)
f = table['offset_hz']
sources = [k.removesuffix('_rad2_per_hz_one_sided') for k in table.dtype.names
           if k.endswith('_rad2_per_hz_one_sided') and not k.startswith('total_')]
fig, axes = plt.subplots(3, 1, figsize=(8, 10), layout='constrained')
for name in sources:
    axes[0].semilogx(f, 10*np.log10(table[name + '_rad2_per_hz_one_sided']/2), label=name)
    gain = table[name + '_power_gain']
    axes[1].semilogx(f, 10*np.log10(gain/np.interp(100_000, f, gain)), label=name)
axes[0].semilogx(f, table['total_ssb_dbc_per_hz'], 'k', label='total')
axes[0].set(xlabel='Offset (Hz)', ylabel='SSB (dBc/Hz)', title='Teaching source budget')
axes[1].set(xlabel='Offset (Hz)', ylabel='Gain relative to 100 kHz (dB)', title='Transfer shapes')
axes[0].legend(ncol=2)
axes[1].legend(ncol=2)
markers = evm['measurement']['markers']
fit = [m for m in markers if m['split'] == 'fit']
held = evm['validation']['holdouts']
axes[2].semilogx([m['offset_hz'] for m in fit], [m['ssb_dbc_per_hz'] for m in fit], 'o-', label='Fit markers')
axes[2].semilogx([m['offset_hz'] for m in held], [m['ssb_dbc_per_hz'] for m in held], 's', label='Measured holdout')
axes[2].semilogx([m['offset_hz'] for m in held], [m['prediction_dbc_per_hz'] for m in held], 'x', label='Prediction')
axes[2].set(xlabel='Offset (Hz)', ylabel='SSB (dBc/Hz)', title='TI SNAU210B Figure 5-1: 14 GHz measurement')
axes[2].legend()
fig.savefig(out / 'noise01-evidence.png', dpi=120)
plt.close(fig)
print('Saved spectrum.csv, summary.json, fft-check.json, evm.json, and noise01-evidence.png')

Saved spectrum.csv, summary.json, fft-check.json, evm.json, and noise01-evidence.png
